# Growth Lab Project 2026 - FIFA World Cup
#### - Sophia Durham & Hafsa Hussain

> "What drives success for teams in the FIFA World Cup and what factors influence match outcomes?”​

## Importing Libraries

In [1]:
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical operations
import plotly.express as px  # For interactive plots
import plotly.graph_objects as go  # For custom charts
from plotly.subplots import make_subplots
import seaborn as sns  # For statistical visualizations
import re  # For advanced string operations

# Configure display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 20)

print("All libraries imported successfully!")

All libraries imported successfully!


## Loading Data

In [2]:
df_team_attacking = pd.read_csv('../data/wc2026_team_attacking.csv')
df_team_defending = pd.read_csv('../data/wc2026_team_defending.csv')
df_team_discipline = pd.read_csv('../data/wc2026_team_discipline.csv')
df_team_distribution = pd.read_csv('../data/wc2026_team_distribution.csv')
df_team_goalkeeping = pd.read_csv('../data/wc2026_team_goalkeeping.csv')
df_team_movement = pd.read_csv('../data/wc2026_team_movement.csv')
df_team_physical = pd.read_csv('../data/wc2026_team_physical.csv')


## First Analysis of Datasets

### Provided datasets
1. https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017/data?select=shootouts.csv
2. https://www.kaggle.com/datasets/kulkarniparth09/fifa-world-cup-complete-dataset-19302026?select=wc_top_scorers.csv

### 2026 World Cup Datasets
1. https://www.kaggle.com/datasets/colinc88/fifa-world-cup-2026-complete-team-statistics?select=wc2026_team_physical.csv

The following datasets are sourced from Kaggle (see above), and represent the data found on the official FIFA World Cup 2026 website. They provide us with team-level data for the 2026 World Cup, and give us more insight into the most recent team strategy, match outcomes, and gameplay.

* wc2026_team_attacking.csv
* wc2026_team_defending.csv
* wc2026_team_discipline.csv
* wc2026_team_distribution.csv
* wc2026_team_goalkeeping.csv
* wc2026_team_movement.csv
* wc2026_team_physical.csv

#### Creating Cleaning Copies
Separate copies of the raw DataFrames are created before cleaning.

This ensures that the originally imported data remains available for comparison and that cleaning operations do not accidentally alter the raw DataFrames.

In [3]:
attacking_clean = df_team_attacking.copy()
defending_clean = df_team_defending.copy()
discipline_clean = df_team_discipline.copy()
distribution_clean = df_team_distribution.copy()
goalkeeping_clean = df_team_goalkeeping.copy()
movement_clean = df_team_movement.copy()
physical_clean = df_team_physical.copy()

In [4]:
def dataset_overview(df, name):
    print(f"===== {name} =====")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print()
    
    print("Data types:")
    print(df.dtypes)
    print()
    
    print("Missing values:")
    print(df.isna().sum())
    print()
    
    print("Missing percentages:")
    print((df.isna().mean() * 100).round(2))
    print()
    
    print("Exact duplicate rows:")
    print(df.duplicated().sum())

#### Cleaning `wc2026_team_attacking.csv`
This dataset provides us with attacking metrics for 2026 FIFA World Cup Teams. Each row represents one team, and includes data such as how many goals were scored overall, how many attempts at goal, how much possession of the ball was had etc. 

In [5]:
dataset_overview(attacking_clean, "Team Attacking Dataset")
attacking_clean.head()

===== Team Attacking Dataset =====
Rows: 48
Columns: 14

Data types:
Team                                  str
Goals                               int64
Assists                             int64
AttemptsAtGoal                      int64
AttemptsOnTarget                    int64
AttemptsOffTarget                   int64
AttemptsAtGoalConversionRatePC      int64
AttemptsInsideThePenaltyArea        int64
AttemptsOutsideThePenaltyArea       int64
HeadedAttemptsAtGoal                int64
xG                                float64
xGEfficiency                          str
Corners                             int64
PossessionControlPC                 int64
dtype: object

Missing values:
Team                              0
Goals                             0
Assists                           0
AttemptsAtGoal                    0
AttemptsOnTarget                  0
AttemptsOffTarget                 0
AttemptsAtGoalConversionRatePC    0
AttemptsInsideThePenaltyArea      0
AttemptsOutsideThePenalt

,Team,Goals,Assists,AttemptsAtGoal,AttemptsOnTarget,AttemptsOffTarget,AttemptsAtGoalConversionRatePC,AttemptsInsideThePenaltyArea,AttemptsOutsideThePenaltyArea,HeadedAttemptsAtGoal,xG,xGEfficiency,Corners,PossessionControlPC
0,England,20,14,118,52,40,17,88,30,27,15.03,1.33x,40,50
1,France,20,18,139,59,59,14,77,62,10,16.78,1.19x,51,51
2,Argentina,19,12,115,46,49,17,66,49,22,15.45,1.23x,41,53
3,Spain,14,10,140,54,59,10,79,61,22,17.48,0.8x,54,58
4,Belgium,14,10,112,35,43,13,74,38,16,10.62,1.32x,24,46


It can be seen that there is no missing data and there are no exact duplicate rows. 
It's worth checking that all the teams are unique to check that there aren't multiple entries for the same team, with different values for other columns.

In [6]:
print(f'Total Rows: {attacking_clean.shape[0]}')
print(f'Unique Teams: {attacking_clean["Team"].nunique()}')


Total Rows: 48
Unique Teams: 48


##### Checking Impossible Values in Percentage Columns
The columns `AttemptsAtGoalConversionRatePC ` and `PossessionControlPC` both contain percentage values, so it's important to verify that none of the values are impossible i.e. are between 0 - 100.

In [7]:
attacking_clean[['AttemptsAtGoalConversionRatePC', 'PossessionControlPC']].describe()

,AttemptsAtGoalConversionRatePC,PossessionControlPC
count,48.000000,48.000000
mean,11.229167,44.333333
std,5.482039,8.213464
min,0.000000,27.000000
25%,6.000000,37.000000
50%,11.500000,46.000000
75%,14.000000,51.000000
max,25.000000,58.000000


In [8]:
impossible_conversion_rate = (attacking_clean['AttemptsAtGoalConversionRatePC'][
    (attacking_clean['AttemptsAtGoalConversionRatePC'].dropna() > 100)
    |
    (attacking_clean['AttemptsAtGoalConversionRatePC'].dropna() < 0)
]
).sum()

impossible_possession_control = (attacking_clean['PossessionControlPC'][
    (attacking_clean['PossessionControlPC'].dropna() > 100)
    |
    (attacking_clean['PossessionControlPC'].dropna() < 0)
]
).sum()

print("Impossible AttemptsAtGoalConversionRatePC values:", impossible_conversion_rate)
print("Impossible PossessionControlPC values:", impossible_possession_control)


Impossible AttemptsAtGoalConversionRatePC values: 0
Impossible PossessionControlPC values: 0


Neither of these columns contain impossible percentage values so do not need investigating.

##### Converting xG Efficiency to float type

The xG efficiency score (a measure of finishing efficiency, calculated as Goals divided by Expected Goals (XG), indicating how well a team converts chances relative to the quality of those chances) is essentially a coefficient value, which we can use to measure a team's attacking quality if it is converted from a string to a float type.

In [9]:
attacking_clean['xGEfficiency'] = attacking_clean['xGEfficiency'].str.replace('x', '').astype(float)
attacking_clean['xGEfficiency'].dtype

dtype('float64')

##### Checking all goals/assists/attempts at goal are non-negative values

In [10]:
attacking_clean[(attacking_clean[['Goals', 'Assists', 'AttemptsAtGoal', 'AttemptsOnTarget', 'AttemptsOffTarget', 'AttemptsInsideThePenaltyArea', 'AttemptsOutsideThePenaltyArea', 'HeadedAttemptsAtGoal', 'xG', 'Corners']] < 0).any(axis=1)].sum()

Team                                 
Goals                               0
Assists                             0
AttemptsAtGoal                      0
AttemptsOnTarget                    0
AttemptsOffTarget                   0
AttemptsAtGoalConversionRatePC      0
AttemptsInsideThePenaltyArea        0
AttemptsOutsideThePenaltyArea       0
HeadedAttemptsAtGoal                0
xG                                0.0
xGEfficiency                      0.0
Corners                             0
PossessionControlPC                 0
dtype: object

##### Checking Attempts at Goal are Inside/Outside the penalty area
Verify that the sum of `AttemptsInsideThePenaltyArea` and `AttemptsOutsideThePenaltyArea` is equal to the total `AttemptsAtGoal`, since a goal can only either be inside or outside the penalty area to be a valid attempt at goal. 

In [11]:
# Verify that the sum of `AttemptsInsideThePenaltyArea` and `AttemptsOutsideThePenaltyArea` is equal to the total `AttemptsAtGoal`
invalid_attempts = (attacking_clean['AttemptsInsideThePenaltyArea'] + attacking_clean['AttemptsOutsideThePenaltyArea'] != attacking_clean['AttemptsAtGoal']).sum()

print("Invalid AttemptsAtGoal values:", invalid_attempts)

Invalid AttemptsAtGoal values: 0


##### Standardising Text Whitespace

In [12]:
attacking_clean['Team'] = attacking_clean['Team'].str.strip()

#### Cleaning `wc2026_team_defending.csv`
In this dataset, each row represents one team and their defending capabilities during the 2026 FIFA World Cup, including:
* how many 'own goals' were scored;
* how many goals were conceded;
* how many forced turnovers of the ball were initiated;
* how many defensive pressures applied against teams;
* how many defensive pressures applied against the player in possesion of the ball

This dataset will be useful to understand the defensive strategies of the teams, and how aggressive they were to re-possess the ball.

In [13]:
dataset_overview(defending_clean, "Team Defending Dataset")
defending_clean.head()

===== Team Defending Dataset =====
Rows: 48
Columns: 7

Data types:
Team                                     str
OwnGoals                               int64
GoalsConceded                          int64
ForcedTurnovers                        int64
BallRecoveryTime                     float64
DefensivePressuresApplied              int64
DefensivePressuresDirectlyApplied      int64
dtype: object

Missing values:
Team                                 0
OwnGoals                             0
GoalsConceded                        0
ForcedTurnovers                      0
BallRecoveryTime                     0
DefensivePressuresApplied            0
DefensivePressuresDirectlyApplied    0
dtype: int64

Missing percentages:
Team                                 0.0
OwnGoals                             0.0
GoalsConceded                        0.0
ForcedTurnovers                      0.0
BallRecoveryTime                     0.0
DefensivePressuresApplied            0.0
DefensivePressuresDirectlyApplie

,Team,OwnGoals,GoalsConceded,ForcedTurnovers,BallRecoveryTime,DefensivePressuresApplied,DefensivePressuresDirectlyApplied
0,Qatar,2,10,101,60.48,832,122
1,Egypt,2,7,224,82.81,1360,238
2,Uzbekistan,1,11,135,50.00,944,142
3,Iraq,1,12,122,61.26,851,155
4,Switzerland,1,6,285,84.65,1581,281


##### Verify numeric columns are non-negative
All numeric columns in this dataset must be non-negative for them to be valid and correct.

In [14]:
defending_clean.describe()

,OwnGoals,GoalsConceded,ForcedTurnovers,BallRecoveryTime,DefensivePressuresApplied,DefensivePressuresDirectlyApplied
count,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000
mean,0.291667,6.416667,179.875000,71.584167,1134.104167,189.395833
std,0.544150,2.959538,66.389446,21.353168,357.199285,64.041788
min,0.000000,1.000000,101.000000,29.940000,490.000000,86.000000
25%,0.000000,4.000000,132.750000,55.907500,870.250000,146.750000
50%,0.000000,6.000000,166.500000,70.255000,1070.500000,172.000000
75%,0.250000,9.000000,193.250000,83.090000,1259.500000,214.500000
max,2.000000,12.000000,399.000000,128.750000,2054.000000,349.000000


In [15]:
defending_negative_values = defending_clean[(defending_clean[['OwnGoals', 'GoalsConceded', 'ForcedTurnovers', 'BallRecoveryTime', 'DefensivePressuresApplied', 'DefensivePressuresDirectlyApplied']] < 0)].sum()

print("Negative values in defending dataset:", defending_negative_values)

Negative values in defending dataset: Team                                    
OwnGoals                             0.0
GoalsConceded                        0.0
ForcedTurnovers                      0.0
BallRecoveryTime                     0.0
DefensivePressuresApplied            0.0
DefensivePressuresDirectlyApplied    0.0
dtype: object


No negative values have been observed so further investigation not neccessary.

##### Standardising Text Whitespace

In [16]:
defending_clean['Team'] = defending_clean['Team'].str.strip()

##### Creating `DirectDefensivePressurePC` column
Looking at the percentage of the Direct Defensive Pressures Applied vs the general Defensive Pressures Applied will give us an indication of how aggressive the defensive strategies were per team. i.e. the higher the percentage, the more aggressive the defensive strategy.

In [17]:
defending_clean['DirectDefensivePressurePC'] = ((defending_clean['DefensivePressuresDirectlyApplied'] / defending_clean['DefensivePressuresApplied']) * 100).round(1)

defending_clean['DirectDefensivePressurePC'].head()

0    14.7
1    17.5
2    15.0
3    18.2
4    17.8
Name: DirectDefensivePressurePC, dtype: float64

##### Final view of `defending_clean` dataset

In [18]:
defending_clean.describe()

,OwnGoals,GoalsConceded,ForcedTurnovers,BallRecoveryTime,DefensivePressuresApplied,DefensivePressuresDirectlyApplied,DirectDefensivePressurePC
count,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000
mean,0.291667,6.416667,179.875000,71.584167,1134.104167,189.395833,16.810417
std,0.544150,2.959538,66.389446,21.353168,357.199285,64.041788,2.575683
min,0.000000,1.000000,101.000000,29.940000,490.000000,86.000000,12.600000
25%,0.000000,4.000000,132.750000,55.907500,870.250000,146.750000,15.000000
50%,0.000000,6.000000,166.500000,70.255000,1070.500000,172.000000,16.600000
75%,0.250000,9.000000,193.250000,83.090000,1259.500000,214.500000,17.525000
max,2.000000,12.000000,399.000000,128.750000,2054.000000,349.000000,25.100000


#### Cleaning `wc2026_team_discipline.csv`
In this dataset, each row represents one team and their disciplinary behaviours during the 2026 FIFA World Cup, such as fouling behaviour, yellow and red cards, and offsides. Where the red card was given in place of 2 yellow cards, it is also indicated in the `IndirectRedCards` column.

In [19]:
dataset_overview(discipline_clean, "Team Discipline Dataset")
discipline_clean.head()

===== Team Discipline Dataset =====
Rows: 48
Columns: 7

Data types:
Team                  str
FoulsAgainst        int64
FoulsFor            int64
YellowCards         int64
RedCards            int64
IndirectRedCards    int64
Offsides            int64
dtype: object

Missing values:
Team                0
FoulsAgainst        0
FoulsFor            0
YellowCards         0
RedCards            0
IndirectRedCards    0
Offsides            0
dtype: int64

Missing percentages:
Team                0.0
FoulsAgainst        0.0
FoulsFor            0.0
YellowCards         0.0
RedCards            0.0
IndirectRedCards    0.0
Offsides            0.0
dtype: float64

Exact duplicate rows:
0


,Team,FoulsAgainst,FoulsFor,YellowCards,RedCards,IndirectRedCards,Offsides
0,Argentina,113,100,15,1,1,22
1,Spain,101,89,6,0,0,21
2,Switzerland,87,89,8,1,1,13
3,France,84,74,6,0,0,14
4,England,81,116,8,1,0,15


##### Standardising Text Whitespace

In [20]:
discipline_clean['Team'] = discipline_clean['Team'].str.strip()

##### Verifying that numeric columns are non-negative
It's impossible for the count of fouls, red/yellow cards and offsides to be negative, so each column will be checked for the presense of negative values.

In [21]:
discipline_negative_values = discipline_clean[(discipline_clean[['FoulsAgainst', 'FoulsFor', 'YellowCards', 'RedCards', 'IndirectRedCards', 'Offsides']] < 0)].sum()

print("Negative values in discipline dataset:", discipline_negative_values)

Negative values in discipline dataset: Team                   
FoulsAgainst        0.0
FoulsFor            0.0
YellowCards         0.0
RedCards            0.0
IndirectRedCards    0.0
Offsides            0.0
dtype: object


No negative values identified, so no further investigation required.

##### Verification on `IndirectRedCards`
An 'indirect red card' is given when a player receives 2 yellow cards. It is therefore important to check that where an 'indirect red card' has been counted, that there have been at least 2 yellow cards for that team.

In [22]:
indirect_red_cards_invalid = discipline_clean[(discipline_clean['IndirectRedCards'] > 0) & (discipline_clean['YellowCards'] < (2 * discipline_clean['IndirectRedCards']))]

print("Teams with invalid Indirect Red Cards:", indirect_red_cards_invalid.shape[0])


Teams with invalid Indirect Red Cards: 0


##### Creating Column for `IndirectRedCardPC`
This column will indicate what proportion of the Red Cards given were 'Indirect Red Cards'.

In [23]:
discipline_clean['IndirectRedCardPC'] = ((discipline_clean['IndirectRedCards'] / discipline_clean['RedCards']) * 100).round(1)
Impossible_RedCardPC_values = discipline_clean['IndirectRedCardPC'][(discipline_clean['IndirectRedCardPC'] < 0) | (discipline_clean['IndirectRedCardPC'] > 100)].dropna().sum()

print("Teams with impossible IndirectRedCardPC values:", Impossible_RedCardPC_values)
discipline_clean.head()

Teams with impossible IndirectRedCardPC values: 0.0


,Team,FoulsAgainst,FoulsFor,YellowCards,RedCards,IndirectRedCards,Offsides,IndirectRedCardPC
0,Argentina,113,100,15,1,1,22,100.0
1,Spain,101,89,6,0,0,21,NaN
2,Switzerland,87,89,8,1,1,13,100.0
3,France,84,74,6,0,0,14,NaN
4,England,81,116,8,1,0,15,0.0


Where the value for `IndirectRedCardPC` is NaN, it is because there have been no red cards attributed to that team, therefore dividing by 0, which is impossible. These null values will be kept to distinguish between the rows where red cards are given but they are not 'indirect' (0 value) vs. the rows where no red cards have been given at all (NaN value)

#### Cleaning `wc2026_team_distribution.csv`
In this dataset, each row represents a team and their passing and ball distribution statistics for 2026 FIFA World Cup, looking at passing, crossing, switching of play, and defensive linebreaks, as well as the accuracy of these.

In [24]:
print(dataset_overview(distribution_clean, "Team Distribution Dataset"))
distribution_clean.head()

===== Team Distribution Dataset =====
Rows: 48
Columns: 11

Data types:
Team                              str
Passes                          int64
PassesCompleted                 int64
PassingAccuracyPC               int64
Crosses                         int64
CrossingAccuracyPC              int64
TakeOnsCompleted                int64
DefensiveLineBreaksAttempted    int64
DefensiveLineBreaksAccPC        int64
SwitchesOfPlayAttempted         int64
SwitchesOfPlayAccPC             int64
dtype: object

Missing values:
Team                            0
Passes                          0
PassesCompleted                 0
PassingAccuracyPC               0
Crosses                         0
CrossingAccuracyPC              0
TakeOnsCompleted                0
DefensiveLineBreaksAttempted    0
DefensiveLineBreaksAccPC        0
SwitchesOfPlayAttempted         0
SwitchesOfPlayAccPC             0
dtype: int64

Missing percentages:
Team                            0.0
Passes                          0.

,Team,Passes,PassesCompleted,PassingAccuracyPC,Crosses,CrossingAccuracyPC,TakeOnsCompleted,DefensiveLineBreaksAttempted,DefensiveLineBreaksAccPC,SwitchesOfPlayAttempted,SwitchesOfPlayAccPC
0,Spain,5470,4945,90,178,20,74,206,66,53,96
1,Argentina,5248,4699,90,111,32,58,138,57,39,95
2,France,4318,3864,89,129,16,87,170,66,41,93
3,England,4014,3577,89,175,27,85,207,66,48,81
4,Morocco,3665,3240,88,99,24,66,110,52,26,85


##### Standardising Text Whitespace

In [25]:
distribution_clean['Team'] = distribution_clean['Team'].str.strip()

##### Verify that numeric columns are non-negative
It is impossible for the metrics in the dataset to be a negative value, so all numeric columns will be checked for the existence of any negative values

In [26]:
impossible_distribution_values = distribution_clean[(distribution_clean[['Passes', 'PassesCompleted', 'PassingAccuracyPC', 'Crosses', 'CrossingAccuracyPC', 'TakeOnsCompleted', 'DefensiveLineBreaksAttempted', 'DefensiveLineBreaksAccPC', 'SwitchesOfPlayAttempted', 'SwitchesOfPlayAccPC']] < 0)].dropna().sum()

print("Negative values in distribution dataset:", impossible_distribution_values)

Negative values in distribution dataset: Team                               
Passes                          0.0
PassesCompleted                 0.0
PassingAccuracyPC               0.0
Crosses                         0.0
CrossingAccuracyPC              0.0
TakeOnsCompleted                0.0
DefensiveLineBreaksAttempted    0.0
DefensiveLineBreaksAccPC        0.0
SwitchesOfPlayAttempted         0.0
SwitchesOfPlayAccPC             0.0
dtype: object


##### Verify all Percentage Columns are between 0 - 100

In [27]:
distribution_PC = [col for col in distribution_clean.columns if 'PC' in col]

impossible_distribution_PC_values = distribution_clean[distribution_PC][(distribution_clean[distribution_PC] < 0) | (distribution_clean[distribution_PC] > 100)].dropna().sum() 

print("Impossible percentage values in distribution dataset:\n", impossible_distribution_PC_values)

Impossible percentage values in distribution dataset:
 PassingAccuracyPC           0.0
CrossingAccuracyPC          0.0
DefensiveLineBreaksAccPC    0.0
SwitchesOfPlayAccPC         0.0
dtype: float64


All percentage columns have been validated as being positive and between 0 - 100. No further investigation needed.

##### Validate `PassingAccuracyPC`
`PassingAccuracyPC` is the percentage of `Passes` that were successfull completed (`PassesCompleted`). Therefore I wasnt to ensure that the percentage is as expected.

In [28]:
incorrect_passing_filter = (distribution_clean['PassesCompleted'] / distribution_clean['Passes']).round(2) != distribution_clean['PassingAccuracyPC']/100

incorrect_passing_accuracy = incorrect_passing_filter.sum()

print('Incorrect Passing Accuracy PC values:', incorrect_passing_accuracy)


Incorrect Passing Accuracy PC values: 0


All PassingAccuracyPC values are correct, therefore no further investigation neccessary.

#### Cleaning `wc2026_team_goalkeeping.csv`
In this dataset, each row represents a team and their goalkeeping performance for 2026 FIFA World Cup.

In [29]:
print(dataset_overview(goalkeeping_clean, "Team Goalkeeping Dataset"))
goalkeeping_clean.head()

===== Team Goalkeeping Dataset =====
Rows: 48
Columns: 6

Data types:
Team                                        str
CleanSheets                               int64
GoalsCondeded                             int64
GoalkeeperSaves                           int64
GoalkeeperActionsInsideThePenaltyArea     int64
GoalkeeperActionsOutsideThePenaltyArea    int64
dtype: object

Missing values:
Team                                      0
CleanSheets                               0
GoalsCondeded                             0
GoalkeeperSaves                           0
GoalkeeperActionsInsideThePenaltyArea     0
GoalkeeperActionsOutsideThePenaltyArea    0
dtype: int64

Missing percentages:
Team                                      0.0
CleanSheets                               0.0
GoalsCondeded                             0.0
GoalkeeperSaves                           0.0
GoalkeeperActionsInsideThePenaltyArea     0.0
GoalkeeperActionsOutsideThePenaltyArea    0.0
dtype: float64

Exact duplicate rows

,Team,CleanSheets,GoalsCondeded,GoalkeeperSaves,GoalkeeperActionsInsideThePenaltyArea,GoalkeeperActionsOutsideThePenaltyArea
0,Spain,7,1,14,120,93
1,France,4,10,15,108,86
2,Colombia,4,1,7,57,54
3,Mexico,4,3,8,60,49
4,Argentina,2,8,20,111,97


##### Standardising Text Whitespace

In [30]:
goalkeeping_clean['Team'] = goalkeeping_clean['Team'].str.strip()

##### Correct typo in column headers

In [31]:
goalkeeping_clean.columns

Index(['Team', 'CleanSheets', 'GoalsCondeded', 'GoalkeeperSaves',
       'GoalkeeperActionsInsideThePenaltyArea',
       'GoalkeeperActionsOutsideThePenaltyArea'],
      dtype='str')

In [32]:
goalkeeping_clean.rename(columns={'GoalsCondeded' : 'GoalsConceded'}, inplace=True)
goalkeeping_clean.columns

Index(['Team', 'CleanSheets', 'GoalsConceded', 'GoalkeeperSaves',
       'GoalkeeperActionsInsideThePenaltyArea',
       'GoalkeeperActionsOutsideThePenaltyArea'],
      dtype='str')

##### Verify numeric columns are non-negative
It's not possible for the numeric columns in this dataset to be negative, so need this needs to be verified.

In [33]:
goalkeeping_clean.describe()

,CleanSheets,GoalsConceded,GoalkeeperSaves,GoalkeeperActionsInsideThePenaltyArea,GoalkeeperActionsOutsideThePenaltyArea
count,48.000000,48.000000,48.000000,48.000000,48.000000
mean,1.145833,6.416667,13.604167,85.395833,71.958333
std,1.429024,2.959538,5.254135,27.496800,22.694979
min,0.000000,1.000000,4.000000,42.000000,34.000000
25%,0.000000,4.000000,9.000000,65.500000,55.500000
50%,1.000000,6.000000,14.000000,81.500000,68.000000
75%,2.000000,9.000000,17.000000,102.000000,88.000000
max,7.000000,12.000000,28.000000,159.000000,132.000000


In [34]:
impossible_goalkeeping_values = goalkeeping_clean[goalkeeping_clean[['CleanSheets', 'GoalsConceded', 'GoalkeeperSaves', 'GoalkeeperActionsInsideThePenaltyArea', 'GoalkeeperActionsOutsideThePenaltyArea']] < 0].dropna().sum()

print("Impossible Goalkeeping values:", impossible_goalkeeping_values)

Impossible Goalkeeping values: Team                                         
CleanSheets                               0.0
GoalsConceded                             0.0
GoalkeeperSaves                           0.0
GoalkeeperActionsInsideThePenaltyArea     0.0
GoalkeeperActionsOutsideThePenaltyArea    0.0
dtype: object


##### Creating `TotalGoalkeeperActions` column, with relative percentage columns
This might be useful later to look at how active the goalkeepers are throughout the tournament

In [35]:
goalkeeping_clean['TotalGoalkeeperActions'] = goalkeeping_clean['GoalkeeperActionsInsideThePenaltyArea'] + goalkeeping_clean['GoalkeeperActionsOutsideThePenaltyArea']

goalkeeping_clean[['GoalkeeperActionsInsideThePenaltyArea','GoalkeeperActionsOutsideThePenaltyArea','TotalGoalkeeperActions' ]].head()

,GoalkeeperActionsInsideThePenaltyArea,GoalkeeperActionsOutsideThePenaltyArea,TotalGoalkeeperActions
0,120,93,213
1,108,86,194
2,57,54,111
3,60,49,109
4,111,97,208


In [36]:
goalkeeping_clean['InsidePenaltyAreaPC'] = ((goalkeeping_clean['GoalkeeperActionsInsideThePenaltyArea'] / goalkeeping_clean['TotalGoalkeeperActions']) * 100 ).round(1)

In [37]:
goalkeeping_clean['OutsidePenaltyAreaPC'] = ((goalkeeping_clean['GoalkeeperActionsOutsideThePenaltyArea'] / goalkeeping_clean['TotalGoalkeeperActions']) * 100 ).round(1)

In [38]:
goalkeeping_clean[['GoalkeeperActionsInsideThePenaltyArea','GoalkeeperActionsOutsideThePenaltyArea','TotalGoalkeeperActions', 'InsidePenaltyAreaPC', 'OutsidePenaltyAreaPC']].head()

,GoalkeeperActionsInsideThePenaltyArea,GoalkeeperActionsOutsideThePenaltyArea,TotalGoalkeeperActions,InsidePenaltyAreaPC,OutsidePenaltyAreaPC
0,120,93,213,56.3,43.7
1,108,86,194,55.7,44.3
2,57,54,111,51.4,48.6
3,60,49,109,55.0,45.0
4,111,97,208,53.4,46.6


In [39]:
goalkeeping_clean.head()

,Team,CleanSheets,GoalsConceded,GoalkeeperSaves,GoalkeeperActionsInsideThePenaltyArea,GoalkeeperActionsOutsideThePenaltyArea,TotalGoalkeeperActions,InsidePenaltyAreaPC,OutsidePenaltyAreaPC
0,Spain,7,1,14,120,93,213,56.3,43.7
1,France,4,10,15,108,86,194,55.7,44.3
2,Colombia,4,1,7,57,54,111,51.4,48.6
3,Mexico,4,3,8,60,49,109,55.0,45.0
4,Argentina,2,8,20,111,97,208,53.4,46.6


#### Cleaning `wc2026_team_movement.csv`
In this dataset, each row represents a team and their off-ball movement and reception behaviours for 2026 FIFA World Cup. The data describes where players are positioned in relation to the opposition players when they signal for the ball and receive the ball.

In [40]:
print(dataset_overview(movement_clean, "Team Movement Dataset"))
movement_clean.head()

===== Team Movement Dataset =====
Rows: 48
Columns: 10

Data types:
Team                                           str
OffersToReceive                              int64
OffersInBehind                               int64
OffersInBetween                              int64
OffersInFront                                int64
OffersInsideTeamShape                        int64
OffersOutsideTeamShape                       int64
ReceptionsInBehind                           int64
ReceptionsBetweenMidfieldAndDefensiveLine    int64
ReceptionsUnderPressure                      int64
dtype: object

Missing values:
Team                                         0
OffersToReceive                              0
OffersInBehind                               0
OffersInBetween                              0
OffersInFront                                0
OffersInsideTeamShape                        0
OffersOutsideTeamShape                       0
ReceptionsInBehind                           0
ReceptionsBetwe

,Team,OffersToReceive,OffersInBehind,OffersInBetween,OffersInFront,OffersInsideTeamShape,OffersOutsideTeamShape,ReceptionsInBehind,ReceptionsBetweenMidfieldAndDefensiveLine,ReceptionsUnderPressure
0,Spain,4036,1134,1543,1359,2367,1669,124,1168,1782
1,Argentina,3283,829,1166,1288,1762,1521,79,873,1614
2,England,3219,796,1099,1324,1837,1382,126,716,1357
3,France,2689,792,900,997,1496,1193,113,900,1405
4,Morocco,2397,679,865,853,1358,1039,61,589,1280


##### Standardise Text Whitespace

In [41]:
movement_clean['Team'] = movement_clean['Team'].str.strip()

##### Verify that all numeric columns are non-negative

In [42]:
impossible_movement_values = (movement_clean[['OffersToReceive', 'OffersInBehind', 'OffersInBetween','OffersInFront', 'OffersInsideTeamShape', 'OffersOutsideTeamShape', 'ReceptionsInBehind', 'ReceptionsBetweenMidfieldAndDefensiveLine', 'ReceptionsUnderPressure']] < 0).dropna().sum()

print(f"Impossible movement values:\n{impossible_movement_values}")

Impossible movement values:
OffersToReceive                              0
OffersInBehind                               0
OffersInBetween                              0
OffersInFront                                0
OffersInsideTeamShape                        0
OffersOutsideTeamShape                       0
ReceptionsInBehind                           0
ReceptionsBetweenMidfieldAndDefensiveLine    0
ReceptionsUnderPressure                      0
dtype: int64


##### Verify offers in different positions are as expected
The total `OffersToReceive` should equate to the sum of `OffersInBehind`, `OffersInBetween` and `OffersInFront`.
<br>
The total `OffersToReceive` should also equate to the sum of `OffersInsideTeamShape` and `OffersOutsideTeamShape`

In [43]:
incorrect_offer_position = (movement_clean['OffersInBehind'] + movement_clean['OffersInBetween'] + movement_clean['OffersInFront'] != movement_clean['OffersToReceive']).dropna().sum()

incorrect_offer_teamshape = (movement_clean['OffersInsideTeamShape'] + movement_clean['OffersOutsideTeamShape'] != movement_clean['OffersToReceive']).dropna().sum()

print("Incorrect Offer to Receive by Position:",incorrect_offer_position)
print("Incorrect Offer to Receive by Team Shape", incorrect_offer_teamshape)

Incorrect Offer to Receive by Position: 0
Incorrect Offer to Receive by Team Shape 0


#### Cleaning `wc2026_team_physical.csv`
In this dataset, each row represents one team and their physical performance metrics for the 2026 FIFA World Cup, such as distance travelled, and speed of running.

In [44]:
print(dataset_overview(physical_clean, "Team Physical Dataset"))
physical_clean.head()

===== Team Physical Dataset =====
Rows: 48
Columns: 5

Data types:
Team                    str
AverageSpeed        float64
HighSpeedRunning      int64
Sprints               int64
TotalDistance       float64
dtype: object

Missing values:
Team                0
AverageSpeed        0
HighSpeedRunning    0
Sprints             0
TotalDistance       0
dtype: int64

Missing percentages:
Team                0.0
AverageSpeed        0.0
HighSpeedRunning    0.0
Sprints             0.0
TotalDistance       0.0
dtype: float64

Exact duplicate rows:
0
None


,Team,AverageSpeed,HighSpeedRunning,Sprints,TotalDistance
0,Czechia,6.33,4330,1381,348826.43
1,Germany,6.29,5991,1990,508926.80
2,Belgium,6.24,8583,3002,739536.58
3,USA,6.23,7252,2446,594895.11
4,New Zealand,6.22,4280,1492,349040.82


##### Standardising Text Whitespace

In [45]:
physical_clean['Team'] = physical_clean['Team'].str.strip()
physical_clean.rename(columns={'AverageSpeed ': 'AverageSpeed'}, inplace=True)

##### Verify that all numeric columns are non-negative

In [46]:
impossible_physical_values = (physical_clean[['AverageSpeed','HighSpeedRunning', 'Sprints', 'TotalDistance']] < 0).dropna().sum()

print(f"Impossible Physical Values:\n{impossible_physical_values}")

Impossible Physical Values:
AverageSpeed        0
HighSpeedRunning    0
Sprints             0
TotalDistance       0
dtype: int64


##### Convert distance to km to maintain consistency with speed (km/h)

In [47]:
physical_clean['TotalDistance'] = physical_clean['TotalDistance']/1000

#### Verify changes to datasets
Dataset sizes and dataset types for original data and cleaned data are compared.

In [48]:
print("===RAW DATA SIZE===")
print("Attacking:", df_team_attacking.shape)
print("Defending:", df_team_defending.shape)
print("Discipline:", df_team_discipline.shape)
print("Distribution:", df_team_distribution.shape)
print("Goalkeeping:", df_team_goalkeeping.shape)
print("Movement:", df_team_movement.shape)
print("Physical:", df_team_physical.shape)

print("\n===CLEANED DATA SIZE===")
print("Attacking:", attacking_clean.shape)
print("Defending:", defending_clean.shape)
print("Discipline:", discipline_clean.shape)
print("Distribution:", distribution_clean.shape)
print("Goalkeeping:", goalkeeping_clean.shape)
print("Movement:", movement_clean.shape)
print("Physical:", physical_clean.shape)

===RAW DATA SIZE===
Attacking: (48, 14)
Defending: (48, 7)
Discipline: (48, 7)
Distribution: (48, 11)
Goalkeeping: (48, 6)
Movement: (48, 10)
Physical: (48, 5)

===CLEANED DATA SIZE===
Attacking: (48, 14)
Defending: (48, 8)
Discipline: (48, 8)
Distribution: (48, 11)
Goalkeeping: (48, 9)
Movement: (48, 10)
Physical: (48, 5)


In [49]:
print("===DATATYPES===")
print("\nATTACKING")
print(attacking_clean.dtypes)

print("\nDEFENDING")
print(defending_clean.dtypes)

print("\nDISCIPLINE")
print(discipline_clean.dtypes)

print("\nDISTRIBUTION")
print(distribution_clean.dtypes)

print("\nGOALKEEPING")
print(goalkeeping_clean.dtypes)

print("\nMOVEMENT")
print(movement_clean.dtypes)

print("\nPHYSICAL")
print(physical_clean.dtypes)

===DATATYPES===

ATTACKING
Team                                  str
Goals                               int64
Assists                             int64
AttemptsAtGoal                      int64
AttemptsOnTarget                    int64
AttemptsOffTarget                   int64
AttemptsAtGoalConversionRatePC      int64
AttemptsInsideThePenaltyArea        int64
AttemptsOutsideThePenaltyArea       int64
HeadedAttemptsAtGoal                int64
xG                                float64
xGEfficiency                      float64
Corners                             int64
PossessionControlPC                 int64
dtype: object

DEFENDING
Team                                     str
OwnGoals                               int64
GoalsConceded                          int64
ForcedTurnovers                        int64
BallRecoveryTime                     float64
DefensivePressuresApplied              int64
DefensivePressuresDirectlyApplied      int64
DirectDefensivePressurePC            float64


## Old - Data Cleaning

### Provided Datasets

In [50]:
# # Create a dictionary to hold all the dataframes for easier access
# prov_dataset_dataframes = {
#     'all_editions': df_all_editions,
#     'all_matches': df_all_matches,
#     'fixtures': df_fixtures,
#     'former_names': df_former_names,
#     'goalscorers': df_goalscorers,
#     'results': df_results,   
#     'shootouts' : df_shootouts,
#     'teams': df_teams,
#     'top_scorers': df_top_scorers
# }    

In [51]:
# # concatenate the 'start date' column and 'year' column and convert to datetime
# df_all_editions['start_date'] = pd.to_datetime(df_all_editions['start_date'] + ' ' + df_all_editions['year'].astype(str), format='%B %d %Y')

# # concatenate the 'end date' column and 'year' column and convert to datetime
# df_all_editions['end_date'] = pd.to_datetime(df_all_editions['end_date'] + ' ' + df_all_editions['year'].astype(str), format='%B %d %Y')

# # check concatenation successful
# print(df_all_editions[['start_date', 'end_date']].head(2))


In [52]:

# # check timelines of World Cup events captured by each dataset
# for name, df in prov_dataset_dataframes.items():
#     for col in df.columns:
#         if not 'date' in col.lower():
#             continue
#         else:            
#             earliest_wc_event = df[col].min()
#             latest_wc_event = df[col].max()
#             print(f"Dataset: {name}, Column: {col}, earliest: {earliest_wc_event.year}, latest: {latest_wc_event.year}")



In [53]:
# # delete rows where date columms are before 1930
# for name, df in prov_dataset_dataframes.items():
#     for col in df.columns:
#         if not 'date' in col.lower() or 'start_date' in col.lower():
#             continue
#         else:
#             df.drop(df[df[col].dt.year < 1930].index, inplace=True)

In [54]:
# # Keep only results from FIFA World Cup tournaments in the results dataset
# df_results = df_results[df_results['tournament'].str.lower() == 'fifa world cup']
# df_results.info()


### FIFA World Cup 2026 Datasets

In [55]:
# dataset_dataframes = {'attacking' : df_team_attacking, 
#                       'defending' : df_team_defending, 
#                       'discipline' : df_team_discipline, 
#                       'distribution' : df_team_distribution, 
#                       'goalkeeping' : df_team_goalkeeping, 
#                       'movement' : df_team_movement, 
#                       'physical' : df_team_physical
#                       }

In [56]:
# Correct team names to align with other datasets
# for name, df in dataset_dataframes.items():
#     df['Team'] = df['Team'].replace({
#         'Congo DR': 'DR Congo',
#         'IR Iran': 'Iran',
#         "Côte d'Ivoire": 'Ivory Coast',
#         "Cabo Verde": "Cape Verde"
#     })